**import spark**

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

**start session**

In [2]:
spark = SparkSession.builder.appName("E-commerce Pipeline").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/25 20:27:34 WARN Utils: Your hostname, cliffe-HP-Pavilion-Laptop-15-cs3xxx, resolves to a loopback address: 127.0.1.1; using 192.168.1.103 instead (on interface wlo1)
26/05/25 20:27:34 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 20:27:35 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


**read the data**

In [4]:
customers_df = spark.read.csv("data/customers.csv", header=True)
items_df = spark.read.csv("data/order_items.csv", header=True)
orders_df = spark.read.csv("data/orders.csv", header=True)
returns_df = spark.read.csv("data/returns.csv", header=True)

**view the data for data type transformation**

1. Customers

In [8]:
customers_df.show()

+-----------+-----------+------------+-------------+--------------------+
|customer_id|signup_date|     country|customer_tier|               email|
+-----------+-----------+------------+-------------+--------------------+
|     C00247| 2018-09-10|       Ghana|       Silver|courtneyberger@ex...|
|     C00125| 2023-12-21|    Ethiopia|         Gold|  abrown@example.com|
|     C00413| 2020-12-13|      Rwanda|     platinum|elizabeth18@examp...|
|     C00219| 2018-11-09|      Rwanda|       BRONZE| ugibson@example.org|
|     C00016| 09/05/2021|     Senegal|         Gold|lynchgeorge@examp...|
|     C00093| 06/03/2023|       Kenya|       BRONZE|michael86@example...|
|     C00223| 07/09/2019|South Africa|     platinum|tylerjohnson@exam...|
|     C00458| 2020-11-16|South Africa|       Bronze| sarah52@example.com|
|     C00059| 26/02/2019|      Rwanda|       Silver|  sara74@example.com|
|     C00356| 2019-02-25|     Senegal|       Silver|carlamoore@exampl...|
|     C00417| 2019-06-14|     Senegal|

In [25]:
customers_df = customers_df.select(F.col("customer_id").cast("string"), F.col("signup_date").cast("date"), F.col("country").cast("string"), F.col("customer_tier").cast("string"), F.col("email").cast("string"))

In [39]:
# Confirm cast
customers_df.show()

26/05/25 21:19:07 ERROR Executor: Exception in task 0.0 in stage 11.0 (TID 11)
org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '09/05/2021' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 1 in cell [26]

	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal(ExecutionErrors.scala:115)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal$(ExecutionErrors.scala:102)
	at org.apache.spark.sql.errors.ExecutionErrors$.invalidInputInCastToDatetimeErrorInternal(ExecutionErrors.scala:252)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeError(ExecutionErrors.scala:92)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeError$(ExecutionErro

DateTimeException: [CAST_INVALID_INPUT] The value '09/05/2021' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 1 in cell [26]


In [ ]:
# Remove stubborn rows
rejected_df = 

2. Order Items

In [12]:
items_df.show()

+----------+--------+----------+--------+----------+-----------+
|   item_id|order_id|product_id|quantity|unit_price|   category|
+----------+--------+----------+--------+----------+-----------+
|I000001866|O0000633|     P0289|       6|    364.01|     beauty|
|I000002153|O0000724|     P0460|       5|    395.26|home_garden|
|I000000203|O0000072|     P0249|       6|    327.38|     beauty|
|I000001503|O0000510|     P0341|       7|    496.84|       toys|
|I000002193|O0000743|     P0318|       1|    426.77|     sports|
|I000001284|O0000434|     P0309|       1|     21.39|home_garden|
|I000000560|O0000196|     P0229|       5|    349.46|   clothing|
|I000003401|O0001149|     P0300|       9|    351.22|       toys|
|I000003511|O0001193|     P0225|       2|     39.28|   clothing|
|I000003976|O0001358|     P0089|       9|    326.18|     sports|
|I000005129|O0001736|     P0304|       4|      83.7|      books|
|I000003561|O0001211|     P0376|       6|    467.11|   clothing|
|I000000341|O0000119|    

In [26]:
items_df = items_df.select(F.col("item_id").cast("string"), F.col("order_id").cast("string"), F.col("product_id").cast("string"), F.col("quantity").cast("int"), F.col("unit_price").cast("double"), F.col("category").cast("string"))

3. Orders

In [14]:
orders_df.show()

+--------+-----------+----------+---------+------------+------------+
|order_id|customer_id|order_date|   status|total_amount|discount_pct|
+--------+-----------+----------+---------+------------+------------+
|O0001478|     C00173|2023-03-29| refunded|     1239.08|        15.0|
|O0000893|     C00194|02/10/2023|  shipped|      858.91|        25.0|
|O0000231|     C00446|2023-12-04|  pending|      1920.8|        20.0|
|O0000466|     C00342|2024-05-27| refunded|     1432.04|        20.0|
|O0000931|     C00036|23/02/2024|  shipped|         9.3|        NULL|
|O0000249|     C00422|2022-01-28|  shipped|     2390.69|        25.0|
|O0001200|     C00375|30/11/2022| refunded|      881.92|         0.0|
|O0001920|     C00049|2024-06-03|  pending|     1034.52|        20.0|
|O0000937|       NULL|2024-03-14|completed|     1842.99|        10.0|
|O0001488|     C00122|2023-05-11|cancelled|     2094.84|        10.0|
|O0001969|     C00242|2024-04-16|cancelled|      1628.3|        20.0|
|O0000800|     C0042

In [27]:
orders_df = orders_df.select(F.col("order_id").cast("string"), F.col("customer_id").cast("string"), F.col("order_date").cast("date"), F.col("status").cast("string"), F.col("total_amount").cast("double"), F.col("discount_pct").cast("double"))

4. Returns

In [18]:
returns_df.show()

+-------------+---------------+-----------+----------------+-------------+
|    return_id|       order_id|return_date|          reason|refund_amount|
+-------------+---------------+-----------+----------------+-------------+
|      R000215|       O0000434| 2023-07-25|      wrong_item|      1141.82|
|      R000234|       O0001069| 2024-05-21|       defective|       180.75|
|      R000268|       O0001933| 2024-04-10|    arrived_late|        331.7|
|      R000181|       O0000885| 2022-04-27|not_as_described|       129.94|
|      R000156|       O0001473| 2023-08-11|       defective|       624.34|
|      R000203|       O0001982| 2022-04-15|    arrived_late|      1084.89|
|      R000017|       O0000643| 2023-06-11|       defective|       412.06|
|      R000204|       O0001279| 2022-05-28|not_as_described|       109.94|
|      R000068|       O0001468| 2023-06-20|       defective|      2175.94|
|      R000029|       O0000159| 07/02/2024|    changed_mind|       386.67|
|      R000194|       O00

In [28]:
returns_df = returns_df.select(F.col("return_id").cast("string"), F.col("order_id").cast("string"), F.col("return_date").cast("date"), F.col("reason").cast("string"), F.col("refund_amount").cast("double"))

**Confirm data type changes**

In [30]:
customers_df.dtypes

[('customer_id', 'string'),
 ('signup_date', 'date'),
 ('country', 'string'),
 ('customer_tier', 'string'),
 ('email', 'string')]

In [31]:
items_df.dtypes

[('item_id', 'string'),
 ('order_id', 'string'),
 ('product_id', 'string'),
 ('quantity', 'int'),
 ('unit_price', 'double'),
 ('category', 'string')]

In [32]:
orders_df.dtypes

[('order_id', 'string'),
 ('customer_id', 'string'),
 ('order_date', 'date'),
 ('status', 'string'),
 ('total_amount', 'double'),
 ('discount_pct', 'double')]

In [33]:
returns_df.dtypes

[('return_id', 'string'),
 ('order_id', 'string'),
 ('return_date', 'date'),
 ('reason', 'string'),
 ('refund_amount', 'double')]

**Drop duplicates**

In [34]:
# Drop duplicate columns based on customer id
customers_df = customers_df.dropDuplicates(['customer_id'])

In [35]:
# Drop duplicate columns based on item id
items_df = items_df.dropDuplicates(['item_id'])

In [38]:
duplicate_orders = orders_df.groupBy(orders_df.columns) \
    .count() \
    .filter(F.col("count") > 1)
duplicate_orders.show()

26/05/25 21:16:40 ERROR Executor: Exception in task 0.0 in stage 10.0 (TID 10)
org.apache.spark.SparkDateTimeException: [CAST_INVALID_INPUT] The value '02/10/2023' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 1 in cell [28]

	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal(ExecutionErrors.scala:115)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeErrorInternal$(ExecutionErrors.scala:102)
	at org.apache.spark.sql.errors.ExecutionErrors$.invalidInputInCastToDatetimeErrorInternal(ExecutionErrors.scala:252)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeError(ExecutionErrors.scala:92)
	at org.apache.spark.sql.errors.ExecutionErrors.invalidInputInCastToDatetimeError$(ExecutionErro

DateTimeException: [CAST_INVALID_INPUT] The value '02/10/2023' of the type "STRING" cannot be cast to "DATE" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
== DataFrame ==
"cast" was called from
line 1 in cell [28]
